# 03 Gold aggregates
Builds analysis-ready tables from Silver. Reads only from Silver, never Bronze.
**Reads:** `silver_games`, `silver_player_box_scores`
**Writes:** `gold_team_standings`, `gold_player_season_stats`, `gold_team_rolling_form`

In [0]:
# --- Setup for the Gold notebook ---
import os, sys
from pyspark.sql import functions as F   # Spark's built-in column functions

# Make our src/ package importable (the notebook lives in notebooks/, so go up one level)
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(repo_root, "src"))

# The quality gate from the Silver notebook, reused here
from nba_pipeline.quality import enforce_checks, DataQualityError
print("ready")

## Team game log and standings
Reshapes one row per game into one row per team per game, then aggregates to standings.

In [0]:
# --- Build a team game log: one row per TEAM per game (nothing is saved) ---

# 1. Read the Silver games table (lazy: this only describes the read)
games = spark.table("workspace.nba.silver_games")

# 2. Home team's view of each game
home = games.select(
    "game_id", "game_date",
    F.col("home_team_id").alias("team_id"),
    F.col("home_team_abbr").alias("team_abbr"),
    F.col("home_team_name").alias("team_name"),
    F.col("away_team_id").alias("opponent_id"),
    F.col("away_team_abbr").alias("opponent_abbr"),
    F.lit(True).alias("is_home"),                  # fixed value for every row here
    F.col("home_score").alias("points_for"),
    F.col("away_score").alias("points_against"),
)

# 3. Away team's view of the same games (mirror image of the above)
away = games.select(
    "game_id", "game_date",
    F.col("away_team_id").alias("team_id"),
    F.col("away_team_abbr").alias("team_abbr"),
    F.col("away_team_name").alias("team_name"),
    F.col("home_team_id").alias("opponent_id"),
    F.col("home_team_abbr").alias("opponent_abbr"),
    F.lit(False).alias("is_home"),
    F.col("away_score").alias("points_for"),
    F.col("home_score").alias("points_against"),
)

# 4. Stack the two views (matched by column NAME) and add a true/false win column
team_games_df = (
    home.unionByName(away)
    .withColumn("win", F.col("points_for") > F.col("points_against"))
)

In [0]:
# --- Build team standings: one row per team (nothing is saved) ---
# Relies on: team_games_df (previous step)

standings_df = (
    # 1. Group the 394 team-game rows into one group per team.
    #    (team_abbr and team_name are the same for every row of a team, so
    #    including them in the grouping just carries them through.)
    team_games_df.groupBy("team_id", "team_abbr", "team_name")
    .agg(
        F.count("*").alias("games_played"),
        # Booleans can't be summed directly, so cast true/false to 1/0 first
        F.sum(F.col("win").cast("int")).alias("wins"),
        F.sum((~F.col("win")).cast("int")).alias("losses"),
        F.sum("points_for").alias("points_for"),
        F.sum("points_against").alias("points_against"),
        # Home and away records: count the rows matching each combination
        F.sum(F.when(F.col("is_home") & F.col("win"), 1).otherwise(0)).alias("home_wins"),
        F.sum(F.when(F.col("is_home") & ~F.col("win"), 1).otherwise(0)).alias("home_losses"),
        F.sum(F.when(~F.col("is_home") & F.col("win"), 1).otherwise(0)).alias("away_wins"),
        F.sum(F.when(~F.col("is_home") & ~F.col("win"), 1).otherwise(0)).alias("away_losses"),
    )
    # 2. Derived columns, computed from the aggregated values
    .withColumn("win_pct", F.round(F.col("wins") / F.col("games_played"), 3))
    .withColumn("point_diff", F.col("points_for") - F.col("points_against"))
    .withColumn("avg_point_diff", F.round(F.col("point_diff") / F.col("games_played"), 1))
)

In [0]:
# --- Quality checks on standings_df, with a gate that blocks the write on failure ---
# Relies on: standings_df (previous step) and enforce_checks (setup cell)

# Helper: counts the rows that BREAK a rule (so 0 means the check passes)
def bad(condition):
    return F.sum(F.when(condition, 1).otherwise(0))

# 1. Row-level checks in ONE aggregation (a single pass over the data)
row = standings_df.agg(
    F.count("*").alias("total_rows"),
    # Every team needs an identifier
    bad(F.col("team_id").isNull() | F.col("team_abbr").isNull()).alias("null_keys"),
    # Every game is a win or a loss
    bad(F.col("wins") + F.col("losses") != F.col("games_played")).alias("wins_losses_ne_games"),
    # Home and away records must add up to the overall record
    bad(F.col("home_wins") + F.col("away_wins") != F.col("wins")).alias("home_away_wins_mismatch"),
    bad(F.col("home_losses") + F.col("away_losses") != F.col("losses")).alias("home_away_losses_mismatch"),
    # A winning percentage is a fraction between 0 and 1
    bad((F.col("win_pct") < 0) | (F.col("win_pct") > 1)).alias("win_pct_out_of_range"),
).first()
assert row is not None, "Standings aggregation returned no row"
results = row.asDict()
total = results.pop("total_rows")

# 2. League-wide totals from the standings table
league = standings_df.agg(
    F.sum("games_played").alias("games_played"),
    F.sum("wins").alias("wins"),
    F.sum("losses").alias("losses"),
    F.sum("points_for").alias("points_for"),
    F.sum("points_against").alias("points_against"),
).first()
assert league is not None, "League totals aggregation returned no row"

# 3. Totals straight from silver_games, bypassing the reshaped game log
silver = spark.table("workspace.nba.silver_games").agg(
    F.count("*").alias("n_games"),
    F.sum("home_score").alias("home_pts"),
    F.sum("away_score").alias("away_pts"),
).first()
assert silver is not None, "Silver totals aggregation returned no row"

# 4. Cross-table checks as 0/1 flags (1 means the rule is broken)
results["team_count_not_30"] = int(total != 30)   # the NBA has 30 teams
results["league_wins_ne_losses"] = int(league["wins"] != league["losses"])
results["league_points_for_ne_against"] = int(league["points_for"] != league["points_against"])
results["games_played_ne_2x_games"] = int(league["games_played"] != 2 * silver["n_games"])
results["points_ne_silver_scores"] = int(league["points_for"] != silver["home_pts"] + silver["away_pts"])

# 5. Gate: print PASS/FAIL per check, then RAISE if any failed (so the next cell never runs)
enforce_checks(results, total, "gold_team_standings")

In [0]:
# --- Write gold_team_standings as a Delta table ---
# Relies on: standings_df. Only run this after the quality gate above has passed.

GOLD_STANDINGS = "workspace.nba.gold_team_standings"

# The ACTION: Spark re-runs the plan (reading Silver, building the game log,
# aggregating) and stores the result. overwrite makes the cell safe to re-run.
(
    standings_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_STANDINGS)
)

# Add a description that shows up in the Catalog UI
spark.sql(f"COMMENT ON TABLE {GOLD_STANDINGS} IS 'Team standings for the 2026-27 season so far: record, home and away splits, and point differential. Aggregated from silver_games.'")

# Verify: read the saved table back and show the top of the standings
print(spark.table(GOLD_STANDINGS).count(), "rows in", GOLD_STANDINGS)
spark.sql(f"""
    SELECT team_abbr, wins, losses, win_pct, avg_point_diff
    FROM {GOLD_STANDINGS}
    ORDER BY win_pct DESC, point_diff DESC
    LIMIT 5
""").show()

## Player season stats
Totals, per-game averages, and shooting percentages for games each player played in.

In [0]:
# --- Build player season stats: one row per player (nothing is saved) ---

# 1. Keep only players who actually played (DNP rows would distort the averages)
box = (
    spark.table("workspace.nba.silver_player_box_scores")
    .where(~F.col("did_not_play"))
)

# 2. Join the game date on from silver_games (inner join on game_id).
#    game_id is unique in silver_games, so this must NOT change the row count.
game_dates = spark.table("workspace.nba.silver_games").select("game_id", "game_date")
played_df = box.join(game_dates, "game_id", "inner")

# 3. Aggregate: group all of a player's games into one row of totals
player_stats_df = (
    played_df.groupBy("player_id")
    .agg(
        F.max_by("player_name", "game_date").alias("player_name"),
        F.max_by("team_abbr", "game_date").alias("team_abbr"),       # team in their latest game
        F.countDistinct("team_id").alias("teams_played_for"),        # > 1 means a mid-season move
        F.count("*").alias("games_played"),
        F.sum(F.col("starter").cast("int")).alias("games_started"),  # true/false -> 1/0, then sum
        F.sum("minutes").alias("total_minutes"),
        F.sum("points").alias("total_points"),
        F.sum("rebounds").alias("total_rebounds"),
        F.sum("assists").alias("total_assists"),
        F.sum("steals").alias("total_steals"),
        F.sum("blocks").alias("total_blocks"),
        F.sum("turnovers").alias("total_turnovers"),
        F.sum("fg_made").alias("total_fg_made"),
        F.sum("fg_attempted").alias("total_fg_attempted"),
        F.sum("fg3_made").alias("total_fg3_made"),
        F.sum("fg3_attempted").alias("total_fg3_attempted"),
        F.sum("ft_made").alias("total_ft_made"),
        F.sum("ft_attempted").alias("total_ft_attempted"),
    )
)

# 4. Per-game averages. This Python loop adds one column definition per pass;
#    it builds the plan, it doesn't process data.
per_game = {
    "mpg": "total_minutes", "ppg": "total_points", "rpg": "total_rebounds",
    "apg": "total_assists", "spg": "total_steals", "bpg": "total_blocks",
    "topg": "total_turnovers",
}
for new_col, total_col in per_game.items():
    player_stats_df = player_stats_df.withColumn(
        new_col, F.round(F.col(total_col) / F.col("games_played"), 1)
    )

# 5. Shooting percentages. try_divide returns NULL (not an error) when there
#    were zero attempts. Written as SQL text inside F.expr.
for pct_col, made, att in [("fg_pct", "total_fg_made", "total_fg_attempted"),
                           ("fg3_pct", "total_fg3_made", "total_fg3_attempted"),
                           ("ft_pct", "total_ft_made", "total_ft_attempted")]:
    player_stats_df = player_stats_df.withColumn(
        pct_col, F.expr(f"round(try_divide({made}, {att}), 3)")
    )

In [0]:
# --- Quality checks on player_stats_df, with a gate that blocks the write on failure ---
# Relies on: player_stats_df (previous step) and enforce_checks (setup cell)
from functools import reduce

# Helper: counts the rows that BREAK a rule (so 0 means the check passes)
def bad(condition):
    return F.sum(F.when(condition, 1).otherwise(0))

# Helper: combine a list of conditions with OR (any column breaks the rule)
def any_of(conds):
    return reduce(lambda a, b: a | b, conds)

total_cols = ["games_played", "games_started", "total_minutes", "total_points", "total_rebounds",
              "total_assists", "total_steals", "total_blocks", "total_turnovers",
              "total_fg_made", "total_fg_attempted", "total_fg3_made", "total_fg3_attempted",
              "total_ft_made", "total_ft_attempted"]

# 1. Row-level checks in ONE aggregation (a single pass over the data)
row = player_stats_df.agg(
    F.count("*").alias("total_rows"),
    # Every player needs an identifier and a name
    bad(F.col("player_id").isNull() | F.col("player_name").isNull()).alias("null_keys"),
    # Totals can never be negative or missing
    bad(any_of([F.col(c).isNull() | (F.col(c) < 0) for c in total_cols])).alias("negative_or_null_totals"),
    # You can't start more games than you played
    bad(F.col("games_started") > F.col("games_played")).alias("started_gt_played"),
    # You can't make more shots than you attempt
    bad((F.col("total_fg_made") > F.col("total_fg_attempted")) |
        (F.col("total_fg3_made") > F.col("total_fg3_attempted")) |
        (F.col("total_ft_made") > F.col("total_ft_attempted"))).alias("made_gt_attempted"),
    # Threes are a subset of field goals
    bad(F.col("total_fg3_made") > F.col("total_fg_made")).alias("threes_exceed_fg"),
    # Season points must follow from the season shooting totals
    bad(F.col("total_points") != 2 * F.col("total_fg_made") + F.col("total_fg3_made") + F.col("total_ft_made")).alias("points_formula_mismatch"),
    # Percentages are fractions between 0 and 1 (nulls, meaning no attempts, are allowed)
    bad((F.col("fg_pct") < 0) | (F.col("fg_pct") > 1) |
        (F.col("fg3_pct") < 0) | (F.col("fg3_pct") > 1) |
        (F.col("ft_pct") < 0) | (F.col("ft_pct") > 1)).alias("pct_out_of_range"),
    # Nobody averages more than 60 minutes a game
    bad(F.col("mpg") > 60).alias("mpg_over_60"),
).first()
assert row is not None, "Player stats aggregation returned no row"
results = row.asDict()
total = results.pop("total_rows")

# 2. Totals from our table
ours = player_stats_df.agg(
    F.sum("games_played").alias("games_played"),
    F.sum("total_points").alias("total_points"),
).first()
assert ours is not None, "Player totals aggregation returned no row"

# 3. The same numbers computed independently from the Silver tables
played_rows = spark.table("workspace.nba.silver_player_box_scores").where(~F.col("did_not_play"))
src = played_rows.agg(
    F.count("*").alias("played_rows"),
    F.countDistinct("player_id").alias("distinct_players"),
).first()
assert src is not None, "Source totals aggregation returned no row"

games = spark.table("workspace.nba.silver_games").agg(
    F.sum("home_score").alias("home_pts"),
    F.sum("away_score").alias("away_pts"),
).first()
assert games is not None, "Game totals aggregation returned no row"

# 4. Cross-table checks as 0/1 flags (1 means the rule is broken)
results["player_count_ne_source"] = int(total != src["distinct_players"])
results["games_played_ne_source_rows"] = int(ours["games_played"] != src["played_rows"])
results["points_ne_silver_scores"] = int(ours["total_points"] != games["home_pts"] + games["away_pts"])

# 5. Gate: print PASS/FAIL per check, then RAISE if any failed (so the next cell never runs)
enforce_checks(results, total, "gold_player_season_stats")

In [0]:
# --- Write gold_player_season_stats as a Delta table ---
# Relies on: player_stats_df. Only run this after the quality gate above has passed.

GOLD_PLAYERS = "workspace.nba.gold_player_season_stats"

# The ACTION: Spark re-runs the plan (read Silver, join, aggregate, add averages)
# and stores the result. overwrite makes the cell safe to re-run.
(
    player_stats_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_PLAYERS)
)

# Add a description that shows up in the Catalog UI
spark.sql(f"COMMENT ON TABLE {GOLD_PLAYERS} IS 'Player season totals, per-game averages and shooting percentages for 2026-27 so far. Only games the player played in. Aggregated from silver_player_box_scores.'")

# Verify 1: read the saved table back and count its rows
print(spark.table(GOLD_PLAYERS).count(), "rows in", GOLD_PLAYERS)

# Verify 2: the player who has played for more than one team
spark.sql(f"""
    SELECT player_name, team_abbr AS latest_team, teams_played_for, games_played
    FROM {GOLD_PLAYERS}
    WHERE teams_played_for > 1
""").show()

# Verify 3: top 5 assist leaders (at least 5 games played)
spark.sql(f"""
    SELECT player_name, team_abbr, games_played, apg
    FROM {GOLD_PLAYERS}
    WHERE games_played >= 5
    ORDER BY apg DESC
    LIMIT 5
""").show()

## Team rolling form
Window functions compute each team's record over its last 10 games, as of every game.
Runs after the standings write because its checks reconcile against `gold_team_standings`.

In [0]:
# --- Build rolling form: one row per team per game (nothing is saved) ---
# Relies on: team_games_df (built earlier in this notebook)
from pyspark.sql import Window

# 1. Two window definitions.
#    w_all:    each team's games in time order (game_id breaks any same-date tie)
#    w_last10: the frame "this game plus the 9 before it", within that same ordering
w_all = Window.partitionBy("team_id").orderBy("game_date", "game_id")
w_last10 = w_all.rowsBetween(-9, Window.currentRow)

# 2. Add the window columns. Every row is KEPT; each new column is computed
#    from a sliding frame of that team's games.
rolling_df = (
    team_games_df
    .withColumn("game_number", F.row_number().over(w_all))                     # 1, 2, 3... per team
    .withColumn("games_in_window", F.count(F.lit(1)).over(w_last10))           # up to 10
    .withColumn("wins_last10", F.sum(F.col("win").cast("int")).over(w_last10))
    .withColumn("losses_last10", F.col("games_in_window") - F.col("wins_last10"))
    .withColumn("ppg_for_last10", F.round(F.avg("points_for").over(w_last10), 1))
    .withColumn("ppg_against_last10", F.round(F.avg("points_against").over(w_last10), 1))
    .withColumn("avg_margin_last10",
                F.round(F.avg(F.col("points_for") - F.col("points_against")).over(w_last10), 1))
)

In [0]:
# --- Quality checks on rolling_df, with a gate that blocks the write on failure ---
# Relies on: rolling_df (previous step), enforce_checks (setup cell), and the
# gold_team_standings table written earlier in this notebook.
from functools import reduce
from pyspark.sql import Window

# Helper: counts the rows that BREAK a rule (so 0 means the check passes)
def bad(condition):
    return F.sum(F.when(condition, 1).otherwise(0))

# Helper: combine a list of conditions with OR (any column breaks the rule)
def any_of(conds):
    return reduce(lambda a, b: a | b, conds)

window_cols = ["game_number", "games_in_window", "wins_last10", "losses_last10",
               "ppg_for_last10", "ppg_against_last10", "avg_margin_last10"]

# 1. Row-level checks in ONE aggregation (a single pass over the data)
row = rolling_df.agg(
    F.count("*").alias("total_rows"),
    # Every row needs its keys
    bad(F.col("team_id").isNull() | F.col("game_id").isNull() | F.col("game_date").isNull()).alias("null_keys"),
    # Comparisons with null aren't caught by the rules below, so check for nulls on their own
    bad(any_of([F.col(c).isNull() for c in window_cols])).alias("null_window_values"),
    # The window should hold min(game_number, 10) games
    bad(F.col("games_in_window") != F.least(F.col("game_number"), F.lit(10))).alias("window_size_wrong"),
    # Wins plus losses must fill the window exactly
    bad(F.col("wins_last10") + F.col("losses_last10") != F.col("games_in_window")).alias("wins_losses_ne_window"),
    # Wins can't be negative or exceed the window
    bad((F.col("wins_last10") < 0) | (F.col("wins_last10") > F.col("games_in_window"))).alias("wins_out_of_range"),
    # NBA teams always score and allow points
    bad((F.col("ppg_for_last10") <= 0) | (F.col("ppg_against_last10") <= 0)).alias("non_positive_ppg"),
    # margin should equal for - against; allow 0.15 because each value was rounded to 1 decimal
    bad(F.abs(F.col("avg_margin_last10") - (F.col("ppg_for_last10") - F.col("ppg_against_last10"))) > 0.15).alias("margin_inconsistent"),
).first()
assert row is not None, "Rolling form aggregation returned no row"
results = row.asDict()
total = results.pop("total_rows")

# 2. No team should have two rows with the same game_number
results["duplicate_game_numbers"] = (
    rolling_df.groupBy("team_id", "game_number").count().where("count > 1").count()
)

# 3. Independent calculation: a RUNNING win total (frame = start of season to this game).
#    While a team has played 10 games or fewer, last-10 wins must equal running wins.
w_running = (
    Window.partitionBy("team_id").orderBy("game_date", "game_id")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)
running = rolling_df.withColumn("running_wins", F.sum(F.col("win").cast("int")).over(w_running))
results["early_wins_ne_running_wins"] = (
    running.where((F.col("game_number") <= 10) & (F.col("wins_last10") != F.col("running_wins"))).count()
)

# 4. Per-team summary: highest game_number and row count. They should be equal
#    (a gap in the numbering would make them differ).
team_counts = rolling_df.groupBy("team_id").agg(
    F.max("game_number").alias("max_game_number"),
    F.count("*").alias("n_rows"),
)
results["game_number_gaps"] = team_counts.where("max_game_number != n_rows").count()

# 5. Reconcile with gold_team_standings: each team's last game_number must equal
#    its games_played there. FULL OUTER join so a missing team on either side shows up.
standings = spark.table("workspace.nba.gold_team_standings").select("team_id", "games_played")
results["standings_games_mismatch"] = (
    team_counts.join(standings, "team_id", "full_outer")
    .where(F.col("max_game_number").isNull() | F.col("games_played").isNull()
           | (F.col("max_game_number") != F.col("games_played")))
    .count()
)

# 6. Row count: exactly two rows (one per team) for every game in silver_games
n_games = spark.table("workspace.nba.silver_games").count()
results["rows_ne_2x_games"] = int(total != 2 * n_games)

# 7. Gate: print PASS/FAIL per check, then RAISE if any failed (so the next cell never runs)
enforce_checks(results, total, "gold_team_rolling_form")

In [0]:
# --- Write gold_team_rolling_form as a Delta table ---
# Relies on: rolling_df. Only run this after the quality gate above has passed.

GOLD_ROLLING = "workspace.nba.gold_team_rolling_form"

# The ACTION: Spark re-runs the plan (read Silver, reshape, apply windows)
# and stores the result. overwrite makes the cell safe to re-run.
(
    rolling_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_ROLLING)
)

# Add a description that shows up in the Catalog UI
spark.sql(f"COMMENT ON TABLE {GOLD_ROLLING} IS 'One row per team per game with the team record and scoring over its last 10 games as of that game (games_in_window is below 10 early in the season). Built from silver_games.'")

# Verify 1: read the saved table back and count its rows
print(spark.table(GOLD_ROLLING).count(), "rows in", GOLD_ROLLING)

# Verify 2: the three most recent rows for Detroit, read from the saved table
spark.sql(f"""
    SELECT game_number, game_date, opponent_abbr, win, wins_last10, losses_last10
    FROM {GOLD_ROLLING}
    WHERE team_abbr = 'DET'
    ORDER BY game_number DESC
    LIMIT 3
""").show()